# Excavating Texts—Data Sources, Mining, and Ownership

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-Digital-Scholarship-Hub/text-and-data/blob/main/02-excavating-texts/Excavating-Texts.ipynb)

In this notebook, we'll explore where and how you can get textual data. We'll use a web **Application Programming Interface (API)**, which is where your computer makes a request for some resource (like data) of a source computer. We will also safely and responsibly **scrape** information from a web page with a Python package called `BeautifulSoup`. These are just a few of the many ways to access textual data and you're always more than welcome to reach out for help if you need it to access data or consider your options. 

> **Important Step**

Run the cells in order from top to bottom. Each cell depends on the ones above it.

A note before we start: this is a shared classroom machine, so don't type or upload anything sensitive or personal.

In [15]:
""" If you're working in Colab, run this cell to bring everything, including the
saved copies of today's data, over from the GitHub repository. It's safe to re-run."""
import os # lets us check which folder we're in
if not os.path.exists('sample_data/manifest.json'): # already in the right folder? then there's nothing to do
    if not os.path.exists('text-and-data'): # only download the repository once
        !git clone https://github.com/UW-Madison-Digital-Scholarship-Hub/text-and-data.git
    %cd text-and-data/02-excavating-texts

In [16]:
# Packages to install if necessary
# Everything we use today comes preinstalled in Colab; this line matters only if you run the notebook elsewhere
!pip install -q requests pandas beautifulsoup4

In [17]:
# Packages to import
import json # reads and writes JavaScript Object Notation (JSON), the format most web APIs speak
import os # builds file paths for our saved copies
import time # lets us pause between requests so we don't overwhelm anyone's server
import requests # sends web requests: the browser, in code
import pandas as pd # turns lists of records into tables
from bs4 import BeautifulSoup # parses HyperText Markup Language (HTML) so we can pull out pieces of a web page
from urllib import robotparser # reads a website's robots.txt rules

### Settings and helpers

All of us are about to send requests to the same servers at the same moment, all from Google's shared network. Servers can slow down, limit how often we can ask, or go offline, so the cell below sets things up to be polite and reliable:

- `USE_CACHE`: set this to `True` to skip the internet entirely and use copies of every response that were saved ahead of time in `sample_data/`.
- `SLEEP_SECONDS`: how long to pause before each request, so we take turns.
- `USER_AGENT`: a short note that tells a server who is asking. Being identifiable is part of being a good citizen on the web.

In [18]:
# Settings for the whole notebook
USE_CACHE = False # True = use the copies saved in sample_data/ instead of the live internet
SLEEP_SECONDS = 2 # seconds to pause before each live request
USER_AGENT = "[YOUR_NAME_PLACEHOLDER] (contact: CONTACT_EMAIL_PLACEHOLDER)" # who's asking
HEADERS = {"User-Agent": USER_AGENT} # requests sends this along with every call

with open("sample_data/manifest.json") as f: # the manifest records where and when each saved copy came from
    MANIFEST = json.load(f)
print(f"{len(MANIFEST)} saved copies available; USE_CACHE is {USE_CACHE}")

14 saved copies available; USE_CACHE is False


In [19]:
def load_saved_copy(cache_name, reason, requested_url=None):
    """Find the copy of a response saved in sample_data/ and explain why we're using it.

    If you changed a search and the live request failed, the saved copy will be for
    the original search, not yours. We say so out loud, because quietly showing
    someone the wrong data is worse than showing them none."""
    if cache_name not in MANIFEST: # nothing was saved under this name
        raise RuntimeError(f"{reason}, and there's no saved copy for this request. Wait a minute and re-run the cell, or ask Brady.")
    entry = MANIFEST[cache_name] # where the saved copy came from, and when
    print(f"{reason}; using copy saved on {entry['retrieved']} from {entry['final_url']}")
    if requested_url and requested_url != entry["final_url"]: # the saved copy answers a different question
        print("Heads up: that saved copy is for the original example, not the request you just made.")
    return os.path.join("sample_data", entry["file"]) # the path of the saved file


def fetch_json(url, params=None, cache_name=None):
    """Ask a web API for JSON data, falling back to a saved copy if we have to.

    We try the live server first, because asking a real server is the skill we're
    practicing. If USE_CACHE is True, or the live request fails for any reason (a
    timeout, an error code, a garbled reply), we load the copy saved in sample_data/
    instead. Either way, the function prints which one you got."""
    requested_url = requests.Request("GET", url, params=params).prepare().url # the full URL, parameters included
    if USE_CACHE:
        reason = "USE_CACHE is True" # we chose not to go online
    else:
        try:
            time.sleep(SLEEP_SECONDS) # take turns with everyone else in the room
            response = requests.get(url, params=params, headers=HEADERS, timeout=15) # give up after 15 seconds
            response.raise_for_status() # treat error codes (404, 429, 503...) as failures
            data = response.json() # parse the JSON text into Python dictionaries and lists
            print(f"Live request succeeded (status {response.status_code}) in {response.elapsed.total_seconds():.1f} seconds")
            return data
        except Exception as error: # anything at all went wrong
            reason = f"Live request failed ({type(error).__name__})"
    with open(load_saved_copy(cache_name, reason, requested_url)) as f: # open the saved copy instead
        return json.load(f)


def fetch_text(url, cache_name, live_allowed=True):
    """Download a text or web page as a string, falling back to a saved copy.

    This works just like fetch_json, except it returns plain text. Some sources ask
    us not to download from them with code at all, and for those we pass
    live_allowed=False so we go straight to the saved copy. We read every file as
    UTF-8, the character encoding that all of today's sources use."""
    if USE_CACHE or not live_allowed:
        reason = "USE_CACHE is True" if USE_CACHE else "Using the saved copy for this source" # say why we're not going online
    else:
        try:
            time.sleep(SLEEP_SECONDS) # take turns with everyone else in the room
            response = requests.get(url, headers=HEADERS, timeout=15) # give up after 15 seconds
            response.raise_for_status() # treat error codes as failures
            print(f"Live request succeeded (status {response.status_code}) in {response.elapsed.total_seconds():.1f} seconds")
            return response.content.decode("utf-8") # turn raw bytes into text
        except Exception as error: # anything at all went wrong
            reason = f"Live request failed ({type(error).__name__})"
    with open(load_saved_copy(cache_name, reason), encoding="utf-8") as f: # open the saved copy instead
        return f.read()

### Where the saved copies came from

Every file in `sample_data/` was retrieved on October 6, 2026. The `manifest.json` file records the exact URL for each one.

| Saved copy | Source |
|---|---|
| `gutendex_*.json` | Responses from [Gutendex](https://gutendex.com/) (`https://gutendex.com/books/`) |
| `hathi_*.json` | Responses from the HathiTrust Bibliographic API (`https://catalog.hathitrust.org/api/volumes/brief/...`) |
| `pg98_raw.txt`, `pg1400_raw.txt`, `pg46_raw.txt` | Project Gutenberg eBooks [#98](https://www.gutenberg.org/ebooks/98), [#1400](https://www.gutenberg.org/ebooks/1400), and [#46](https://www.gutenberg.org/ebooks/46), unmodified, from Project Gutenberg's official mirror (`https://aleph.pglaf.org/cache/epub/...`) |
| `books_toscrape_index.html` | The home page of [books.toscrape.com](https://books.toscrape.com/), a practice site for learning to scrape |

## What Is an API?

An API (Application Programming Interface) is a way for one program to ask another program for information. Many web APIs are just web addresses that return data instead of a web page. Before writing any code, open this link in a new browser tab:

[https://gutendex.com/books/?ids=98](https://gutendex.com/books/?ids=98)

What you see is raw JSON (JavaScript Object Notation): curly braces, quoted labels, and values. It's the catalog record for book #98 in Project Gutenberg. Your browser sent a request and the server sent back data. Now we'll do exactly the same thing in code.

In [20]:
first_url = "https://gutendex.com/books/?ids=98" # the same address you just opened in your browser
try:
    if USE_CACHE: raise RuntimeError("USE_CACHE is True") # skip the live request if we've chosen to
    response = requests.get(first_url, headers=HEADERS, timeout=15) # the browser, in code
    print(f"Status code: {response.status_code}") # 200 means "OK, here's what you asked for"
    print(f"That took {response.elapsed.total_seconds():.1f} seconds\n")
    print(response.text[:500]) # the first 500 characters of the reply: the same JSON your browser showed
except Exception as error: # slow server, no Wi-Fi, or USE_CACHE
    reason = "USE_CACHE is True" if USE_CACHE else f"Live request didn't work ({type(error).__name__})" # say which it was
    with open(load_saved_copy("gutendex_ids_98", reason)) as f:
        print(f.read()[:500]) # the saved copy of the same reply

Status code: 200
That took 0.1 seconds

{"count":1,"next":null,"previous":null,"results":[{"id":98,"title":"A Tale of Two Cities","authors":[{"name":"Dickens, Charles","birth_year":1812,"death_year":1870}],"summaries":["\"A Tale of Two Cities\" by Charles Dickens is a historical novel published in 1859. Set in London and Paris during the tumultuous French Revolution, it follows Dr. Alexandre Manette after his mysterious 18-year imprisonment in the Bastille and his reunion with his daughter Lucie. Their lives become entangled with a Fr


## Open Repositories: Project Gutenberg

[Project Gutenberg](https://www.gutenberg.org/) has been digitizing public-domain books since 1971, mostly through volunteers who type or proofread texts by hand. Project Gutenberg releases its [catalog data](https://www.gutenberg.org/ebooks/offline_catalogs.html) into the public domain, and [Gutendex](https://gutendex.com/) is an independent, open-source project that turns that catalog into a searchable API. Notice that already: the data comes from one organization, and the API from another.

Gutendex takes *parameters*, filters added to the end of the web address. In the cell below, change any of the values and re-run it and the cells after it.

In [21]:
# Your turn: change any of these, then re-run this cell and the ones below it
search = "dickens" # words to look for in titles and author names
languages = "en" # two-letter language codes, separated by commas (e.g. "en,fr")
topic = "" # words to look for in subjects and bookshelves (e.g. "detective"); "" means no filter
author_year_start = 1800 # only authors alive in or after this year
author_year_end = 1899 # ...and in or before this year

search_params = {"search": search, "languages": languages, "topic": topic,
                 "author_year_start": author_year_start, "author_year_end": author_year_end} # bundle them up for requests

In [22]:
%%time
page1 = fetch_json("https://gutendex.com/books/", params=search_params, cache_name="gutendex_search_page1") # ask Gutendex
print(f"Python sees this response as a: {type(page1).__name__}") # JSON objects become Python dictionaries
print(f"Its keys are: {list(page1.keys())}") # the labels at the top level of the JSON
print(f"Books matching the search: {page1['count']}") # how many books matched in total
print(f"Books on this page: {len(page1['results'])}") # Gutendex sends at most 32 at a time
print(f"Address of the next page: {page1['next']}") # None means there isn't one

Live request succeeded (status 200) in 0.1 seconds
Python sees this response as a: dict
Its keys are: ['count', 'next', 'previous', 'results']
Books matching the search: 182
Books on this page: 32
Address of the next page: https://gutendex.com/books/?author_year_end=1899&author_year_start=1800&languages=en&page=2&search=dickens&topic=
CPU times: user 16.9 ms, sys: 4.13 ms, total: 21.1 ms
Wall time: 2.13 s


`results` is a *list* of dictionaries, one per book. Some values are nested: `authors` is itself a list of dictionaries (a book can have several authors), and `formats` is a dictionary whose keys are file types, written as [MIME types](https://en.wikipedia.org/wiki/Media_type) like `text/plain; charset=utf-8`, and whose values are download links.

In [23]:
first_book = page1["results"][0] # the first dictionary in the results list
print(f"Title: {first_book['title']}")
print(f"Authors: {first_book['authors']}") # a list of dictionaries, nested inside our dictionary
print(f"First author's name: {first_book['authors'][0]['name']}") # reach inside: list position 0, then the 'name' key
print("Formats:")
for mime_type, link in first_book["formats"].items(): # each file type and where to download it
    print(f"  {mime_type:35} {link}")

Title: A Tale of Two Cities
Authors: [{'name': 'Dickens, Charles', 'birth_year': 1812, 'death_year': 1870}]
First author's name: Dickens, Charles
Formats:
  text/html                           https://www.gutenberg.org/ebooks/98.html.images
  application/epub+zip                https://www.gutenberg.org/ebooks/98.epub3.images
  application/x-mobipocket-ebook      https://www.gutenberg.org/ebooks/98.kf8.images
  application/rdf+xml                 https://www.gutenberg.org/ebooks/98.rdf
  image/jpeg                          https://www.gutenberg.org/cache/epub/98/pg98.cover.medium.jpg
  application/octet-stream            https://www.gutenberg.org/cache/epub/98/pg98-h.zip
  text/plain; charset=utf-8           https://www.gutenberg.org/ebooks/98.txt.utf-8


Nested JSON is hard to read, so we'll flatten each book into one row of a table, keeping only a few fields. We'll also follow the `next` link once to get a second page of results. We stop at two pages to keep our load on a volunteer-run server small.

In [24]:
%%time
def books_to_table(books):
    """Flatten Gutendex's nested book records into one row per book. We keep only the
    first author, which is a choice: co-authors and editors disappear from the table."""
    rows = []
    for book in books:
        author = book["authors"][0] if book["authors"] else {} # some books have no listed author
        rows.append({"id": book["id"], "title": book["title"], "first_author": author.get("name"),
                     "birth_year": author.get("birth_year"), "death_year": author.get("death_year"),
                     "languages": ", ".join(book["languages"]), "download_count": book["download_count"]})
    return pd.DataFrame(rows).astype({"birth_year": "Int64", "death_year": "Int64"}) # whole numbers, even when a year is missing

all_books = list(page1["results"]) # start with page one
if page1["next"]: # if there's a second page, get it (but stop there)
    page2 = fetch_json(page1["next"], cache_name="gutendex_search_page2")
    all_books += page2["results"]
books_df = books_to_table(all_books) # one row per book
print(f"{len(books_df)} books in our table")
books_df.head(15)

Live request succeeded (status 200) in 0.1 seconds
64 books in our table
CPU times: user 20.5 ms, sys: 16 ms, total: 36.4 ms
Wall time: 2.16 s


,id,title,first_author,birth_year,death_year,languages,download_count
0,98,A Tale of Two Cities,"Dickens, Charles",1812,1870,en,50704
1,47530,"Oliver Twist, Vol. 2 (of 3)","Dickens, Charles",1812,1870,en,42032
2,1400,Great Expectations,"Dickens, Charles",1812,1870,en,38268
3,46,A Christmas Carol in Prose; Being a Ghost Stor...,"Dickens, Charles",1812,1870,en,28618
4,564,The Mystery of Edwin Drood,"Dickens, Charles",1812,1870,en,22977
5,730,Oliver Twist,"Dickens, Charles",1812,1870,en,16142
6,766,David Copperfield,"Dickens, Charles",1812,1870,en,16140
7,1023,Bleak House,"Dickens, Charles",1812,1870,en,16093
8,580,The Pickwick Papers,"Dickens, Charles",1812,1870,en,9151
9,967,Nicholas Nickleby,"Dickens, Charles",1812,1870,en,8293


**Discussion:** Who decided what's in this catalog? Project Gutenberg's texts are mostly works in the public domain in the United States, chosen and prepared by volunteers. Whose books, languages, and kinds of writing are well represented here, and what's missing? What does `download_count` reward?

## HathiTrust: Rights as Data

[HathiTrust](https://www.hathitrust.org/) is a partnership of research libraries that holds millions of digitized volumes, many of them scanned by Google Books. Its Bibliographic API is a *lookup*, not a search: you give it an identifier you already have, such as an ISBN or an OCLC number (a WorldCat record number), and it tells you which copies HathiTrust holds and what you're allowed to do with each one. Try one in your browser: [https://catalog.hathitrust.org/api/volumes/brief/oclc/30822465.json](https://catalog.hathitrust.org/api/volumes/brief/oclc/30822465.json)

We'll look up five titles. Two of them should look familiar from our last workshop:

- *A Tale of Two Cities*, 1859 edition ([the Harvard copy we OCR'd last time](https://babel.hathitrust.org/cgi/pt?id=hvd.32044024216053))
- *A Tale of Two Cities*, 2003 Penguin Classics edition ([ucbk.ark:/28722/h2bc3t44t](https://babel.hathitrust.org/cgi/pt?id=ucbk.ark:/28722/h2bc3t44t))
- *Statistical Abstract of the United States* ([the 1889–1891 volume we used for our table](https://babel.hathitrust.org/cgi/pt?id=coo.31924055969913))
- Rachel Carson, *Silent Spring*, 2002 edition ([hvd.32044126849165](https://babel.hathitrust.org/cgi/pt?id=hvd.32044126849165))
- Toni Morrison, *Beloved*, 1987 ([mdp.49015003142743](https://babel.hathitrust.org/cgi/pt?id=mdp.49015003142743))

In [25]:
%%time
hathi_lookups = [("oclc", "30822465"),       # A Tale of Two Cities (1859)
                 ("isbn", "9780141439600"),  # A Tale of Two Cities (Penguin, 2003)
                 ("oclc", "1193890"),        # Statistical Abstract of the United States
                 ("isbn", "9780618249060"),  # Silent Spring (2002)
                 ("isbn", "9780394535975")]  # Beloved (1987)
hathi_rows = []
for id_type, id_value in hathi_lookups:
    data = fetch_json(f"https://catalog.hathitrust.org/api/volumes/brief/{id_type}/{id_value}.json",
                      cache_name=f"hathi_{id_type}_{id_value}") # one lookup per identifier
    for item in data["items"][:3]: # some records list dozens of copies; three each keeps the table readable
        record = data["records"][item["fromRecord"]] # the catalog record this copy belongs to
        hathi_rows.append({"title": record["titles"][0], "published": record["publishDates"][0],
                           "volume": item["enumcron"] or "", "htid": item["htid"], # htid = HathiTrust's ID for one scanned copy
                           "us_rights": item["usRightsString"], "holding_library": item["orig"]})
hathi_df = pd.DataFrame(hathi_rows)
hathi_df

Live request succeeded (status 200) in 0.1 seconds
Live request succeeded (status 200) in 0.1 seconds
Live request succeeded (status 200) in 0.3 seconds
Live request succeeded (status 200) in 0.1 seconds
Live request succeeded (status 200) in 0.1 seconds
CPU times: user 69.7 ms, sys: 15.8 ms, total: 85.5 ms
Wall time: 10.8 s


,title,published,volume,htid,us_rights,holding_library
0,A tale of two cities,1859,,hvd.32044024216053,Full view,Harvard University
1,A tale of two cities,2003,,ucbk.ark:/28722/h2bc3t44t,Limited (search-only),University of California
2,Statistical abstract of the United States,1878,1st-5th 1878-1882,coo.31924055969780,Full view,Cornell University
3,Statistical abstract of the United States,1878,6th-8th 1883-1885,coo.31924055969772,Full view,Cornell University
4,Statistical abstract of the United States,1878,9th-11th 1886-1888,coo.31924055969764,Full view,Cornell University
5,Silent spring,2002,,hvd.32044126849165,Limited (search-only),Harvard University
6,Beloved : a novel,1987,,mdp.49015003142743,Limited (search-only),University of Michigan
7,Beloved : a novel,1987,,ucbk.ark:/28722/h28k75b8j,Limited (search-only),University of California


"Full view" means anyone in the US can read the full scan online. "Limited (search-only)" means you can search inside the book to see which pages contain a word, but you can't read or download it. Behind each string is a short `rightsCode` (`pd` for public domain, `ic` for in copyright), which records the result of a copyright determination.

**Discussion:** It's the same title, but different permissions. Why would the 1859 *Tale of Two Cities* be fully open while the 2003 edition is search-only, when Dickens wrote both? Who decides, and what could a researcher actually *do* with each row of this table?

## From Catalog to Corpus

A catalog tells us texts exist; a *corpus* is the texts themselves, gathered for analysis. We'll pick three Dickens novels from our Gutendex table and use the `text/plain; charset=utf-8` link in each record's `formats` to see where the plain text lives.

There's a catch worth pausing on. Project Gutenberg's [policy on robot access](https://www.gutenberg.org/policy/robot_access.html) says: *"The Project Gutenberg website is intended for human users only. Any perceived use of automated tools to access the Project Gutenberg website will result in a temporary or permanent block of your IP address."* It points automated tools to its [mirrors](https://www.gutenberg.org/help/mirroring.html) and harvest service instead. All of us downloading from Google's shared network would look exactly like a robot. So we'll use copies saved ahead of time from Project Gutenberg's official mirror, left completely unmodified, Project Gutenberg license and all.

In [26]:
chosen_ids = [98, 1400, 46] # A Tale of Two Cities, Great Expectations, A Christmas Carol
books_by_id = {book["id"]: book for book in all_books} # look up any book from our results by its id
for book_id in chosen_ids:
    book = books_by_id.get(book_id) # None if your search above didn't include this book
    text_link = book["formats"]["text/plain; charset=utf-8"] if book else f"https://www.gutenberg.org/ebooks/{book_id}.txt.utf-8"
    print(f"#{book_id}: {text_link}") # where the catalog says the plain text lives

#98: https://www.gutenberg.org/ebooks/98.txt.utf-8
#1400: https://www.gutenberg.org/ebooks/1400.txt.utf-8
#46: https://www.gutenberg.org/ebooks/46.txt.utf-8


Every Project Gutenberg file wraps the book in a header and a footer (license terms, credits, and metadata) between lines that begin `*** START OF` and `*** END OF`. If we left them in, every book in our corpus would share several thousand words of legal boilerplate, which would skew any counting we do later. So we'll cut them out in code, leaving the saved originals untouched, and save cleaned copies alongside them.

In [27]:
def strip_gutenberg_boilerplate(raw_text):
    """Keep only the text between Project Gutenberg's START and END marker lines.
    If the markers aren't there, return the text unchanged rather than guess."""
    start = raw_text.find("*** START OF") # where the header's last line begins
    end = raw_text.find("*** END OF") # where the footer begins
    if start == -1 or end == -1:
        return raw_text
    start = raw_text.find("\n", start) + 1 # skip to the end of the START line
    return raw_text[start:end].strip() # trim blank lines at either end

In [28]:
%%time
for book_id in chosen_ids:
    url = f"https://www.gutenberg.org/ebooks/{book_id}.txt.utf-8" # the official link, for the record
    raw_text = fetch_text(url, cache_name=f"pg{book_id}_raw", live_allowed=False) # always the saved copy (see the policy above)
    book_text = strip_gutenberg_boilerplate(raw_text) # drop the header and footer
    with open(f"sample_data/pg{book_id}_clean.txt", "w", encoding="utf-8") as f: # save the cleaned version
        f.write(book_text)
    words = book_text.split() # split wherever there's a space or line break
    print(f"#{book_id}: {len(raw_text):,} characters raw, {len(book_text):,} cleaned, {len(words):,} words\n")

Using the saved copy for this source; using copy saved on 2026-10-06 from https://aleph.pglaf.org/cache/epub/98/pg98.txt
#98: 776,912 characters raw, 757,604 cleaned, 135,886 words

Using the saved copy for this source; using copy saved on 2026-10-06 from https://aleph.pglaf.org/cache/epub/1400/pg1400.txt
#1400: 1,013,928 characters raw, 994,616 cleaned, 184,478 words

Using the saved copy for this source; using copy saved on 2026-10-06 from https://aleph.pglaf.org/cache/epub/46/pg46.txt
#46: 177,913 characters raw, 158,414 cleaned, 28,541 words

CPU times: user 17.1 ms, sys: 7.42 ms, total: 24.5 ms
Wall time: 26.7 ms


That `.split()` is the crudest possible tokenizer: it splits wherever there's whitespace, so `"Pip,"` and `"Pip"` count as different words, and `"don't"` stays in one piece. Deciding what counts as a word is a whole set of choices, and it's where our next session, [What's in a Word](../README.md), picks up.

## The Open Web: Scraping

APIs are an invitation: the data provider has decided what to share and in what shape. Scraping is different. It means writing code that reads web pages meant for human eyes and pulls data out of them. We'll practice only on [books.toscrape.com](https://books.toscrape.com/), a fake bookstore built for people learning to scrape. It is **critical** to check whether and how scraping is permitted before actually doing it and, when in doubt, chat with a librarian or avoid doing it. 

The first step is to check the site's `robots.txt` file. This is a convention, not a law: a plain text file at the top of a site that tells automated visitors which pages they may and may not fetch.

Scrapers that ignore the `robots.txt` file have become increasingly common and they put strain on websites and create strong incentives for organizations to make materials inaccessible to protect their infrastructure. 

In [29]:
robots = robotparser.RobotFileParser() # Python's built-in robots.txt reader
try:
    if USE_CACHE: raise RuntimeError("USE_CACHE is True") # this check only makes sense live
    robots_response = requests.get("https://books.toscrape.com/robots.txt", headers=HEADERS, timeout=15)
    print(f"robots.txt status code: {robots_response.status_code}") # 404 means "there's no such file"
    if robots_response.status_code == 404: robots.allow_all = True # no rules published: robotparser treats that as allowed
    else: robots.parse(robots_response.text.splitlines()) # otherwise, read the rules
    print(f"May we fetch the home page? {robots.can_fetch(USER_AGENT, 'https://books.toscrape.com/')}")
except Exception as error:
    reason = "USE_CACHE is True" if USE_CACHE else f"Couldn't check live ({type(error).__name__})" # say which it was
    print(f"{reason}. On October 6, 2026, this site had no robots.txt (status 404).")

robots.txt status code: 404
May we fetch the home page? True


This site has no `robots.txt` at all, and Python's `robotparser` treats that as permission to fetch everything. But *no rules posted* isn't the same as *permission granted*: a site's terms of service, copyright, and the people whose data appears on it all still matter. We'll come back to this in the next section. For now, we'll fetch one page.

In [30]:
page_html = fetch_text("https://books.toscrape.com/", cache_name="books_toscrape_index") # one page, one request
print(f"{len(page_html):,} characters of HTML\n")
print(page_html[:600]) # the start of what your browser receives before it draws the page

Live request succeeded (status 200) in 0.1 seconds
51,274 characters of HTML

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" con


In [31]:
%%time
soup = BeautifulSoup(page_html, "html.parser") # turn the HTML into a structure we can search
scraped_rows = []
for book in soup.select("article.product_pod"): # every book on the page sits inside one of these tags
    scraped_rows.append({"title": book.h3.a["title"], # the full title is stored in the link's title attribute
                         "price": book.select_one("p.price_color").text, # the visible price
                         "rating": book.select_one("p.star-rating")["class"][1]}) # the rating exists only as a CSS class, e.g. "Three"
scraped_df = pd.DataFrame(scraped_rows)
scraped_df.head(10)

CPU times: user 16.2 ms, sys: 2.28 ms, total: 18.5 ms
Wall time: 18.7 ms


,title,price,rating
0,A Light in the Attic,£51.77,Three
1,Tipping the Velvet,£53.74,One
2,Soumission,£50.10,One
3,Sharp Objects,£47.82,Four
4,Sapiens: A Brief History of Humankind,£54.23,Five
5,The Requiem Red,£22.65,One
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,Four
7,The Coming Woman: A Novel Based on the Life of...,£17.93,Three
8,The Boys in the Boat: Nine Americans and Their...,£22.60,Four
9,The Black Maria,£52.15,One


Look at `rating`: on the page it's a row of stars, but in the HTML it's only a style label (`star-rating Three`). Scraping works only because someone happened to encode meaning somewhere a program can find it. Now we'll do what many text-mining pipelines do, which is throw away the HTML and keep only the words:

In [32]:
page_text = " ".join(soup.get_text().split()) # strip every tag and collapse the whitespace
print(f"{len(page_html):,} characters of HTML became {len(page_text):,} characters of text\n")
print(page_text[:600])

51,274 characters of HTML became 1,850 characters of text

All products | Books to Scrape - Sandbox Books to Scrape We love being scraped! Home All products Books Travel Mystery Historical Fiction Sequential Art Classics Philosophy Romance Womens Fiction Fiction Childrens Religion Nonfiction Music Default Science Fiction Sports and Games Add a comment Fantasy New Adult Young Adult Science Poetry Paranormal Art Psychology Autobiography Parenting Adult Fiction Humor Horror History Food and Drink Christian Fiction Business Biography Thriller Contemporary Spirituality Academic Self Help Historical Christian Suspense Short Stories Novels Health Politics Cu


What got stripped? The layout (which words were headings, prices, buttons, or navigation), the images and their star ratings, the links between pages, and any sign of where this text came from or when. That's the theme of this whole series: each step that turns text into data gains us something we can count, and loses context we may not get back.

## Who Decides?

### A responsible-scraping checklist of considerations

Before collecting text from the web, ask:

- **Terms of service:** Does the site's terms of use allow automated access or reuse of its content?
- **robots.txt:** What does the site ask of automated visitors, and are you honoring it?
- **Personal data:** Does the content include information about people, such as names, faces, or posts? Would they expect it to be collected?
- **Purpose:** What are you going to do with it, and could you answer your question with less?
- **Server load:** Are you pausing between requests, identifying yourself, and collecting only what you need?
- **Licensing and copyright:** Who owns the text, and does your use need permission, fall under a license, or rely on fair use?
- **Jurisdiction:** Whose laws apply: yours, the site's, or those of the people in the data? There may be multiple, overlapping jurisdictions for any given site. 
- **Library agreements:** Does your institution already license this content, and does that license permit text and data mining? Ask a librarian.

## Try Your Own

Now it's your turn. Change the search below to anything you like: an author, a title word, or a topic. Then look up a book of your own in HathiTrust by its ISBN, which you can usually find on the copyright page or the back cover, or in a library catalog record.

One rule: **don't point the scraping code at real websites.** Those cells are for books.toscrape.com only. Real sites have terms, owners, and limits, and the checklist above applies.

In [33]:
# Your turn: search Project Gutenberg through Gutendex
my_search = "austen" # an author, a title word, or a topic
my_results = fetch_json("https://gutendex.com/books/", params={"search": my_search, "languages": "en"},
                        cache_name="gutendex_try_your_own") # one page of results
print(f"{my_results['count']} books match '{my_search}'")
books_to_table(my_results["results"]).head(10) #limit results to the first 10

Live request succeeded (status 200) in 0.1 seconds
42 books match 'austen'


,id,title,first_author,birth_year,death_year,languages,download_count
0,1342,Pride and Prejudice,"Austen, Jane",1775,1817,en,190246
1,21839,Sense and Sensibility,"Austen, Jane",1775,1817,en,56986
2,42671,Pride and Prejudice,"Austen, Jane",1775,1817,en,33256
3,31100,The Complete Project Gutenberg Works of Jane A...,"Austen, Jane",1775,1817,en,32900
4,161,Sense and Sensibility,"Austen, Jane",1775,1817,en,32133
5,158,Emma,"Austen, Jane",1775,1817,en,24182
6,1212,Love and Freindship [sic],"Austen, Jane",1775,1817,en,15438
7,105,Persuasion,"Austen, Jane",1775,1817,en,14223
8,121,Northanger Abbey,"Austen, Jane",1775,1817,en,13488
9,141,Mansfield Park,"Austen, Jane",1775,1817,en,8580


In [34]:
# Your turn: look up a book in HathiTrust by ISBN (digits only, no dashes)
my_isbn = "9780679732761" # Ralph Ellison, Invisible Man (1995 edition)
my_data = fetch_json(f"https://catalog.hathitrust.org/api/volumes/brief/isbn/{my_isbn}.json",
                     cache_name="hathi_isbn_9780679732761") # the saved copy is for the example ISBN only
if not my_data["items"]: # HathiTrust doesn't hold this edition
    print(f"HathiTrust has no record for ISBN {my_isbn}. Try another edition's ISBN.")
for item in my_data["items"]: # one line per scanned copy
    print(f"{my_data['records'][item['fromRecord']]['titles'][0]} | {item['usRightsString']} | {item['orig']} | {item['itemURL']}")

Live request succeeded (status 200) in 0.2 seconds
Invisible man | Limited (search-only) | University of Michigan | https://babel.hathitrust.org/cgi/pt?id=mdp.39015051894387
Invisible man | Limited (search-only) | University of California | https://babel.hathitrust.org/cgi/pt?id=ucbk.ark:/28722/h2nv99s22


Want to try a different kind of identifier instead? Copy the cell above and swap `isbn` for `oclc` in the URL, then paste in an OCLC number, which you'll find on any [WorldCat](https://search.worldcat.org/) record. You can also add `topic`, `author_year_start`, or `author_year_end` to the Gutendex search, just like in the parameters cell near the top.

## Further Reading

There are a lot of great resources out there about digitization and textual data. A couple of favorites that remain particularly relevant include: 

Marcum, Deanna B., and Roger C. Schonfeld. [*Along Came Google: A History of Library Digitization.*](https://press.princeton.edu/books/hardcover/9780691172712/along-came-google?srsltid=AU7gw4Vb6oHjEr-OcwjRFCJO2EEFxXvPc450R_HmbTi3kLJxYgwJ5Koj) Princeton University Press, 2023.

Thompson, John B. [*Book Wars: The Digital Revolution in Publishing.*](https://www.politybooks.com/bookdetail?book_slug=book-wars-the-digital-revolution-in-publishing--9781509546787) Polity Press, 2021.


> **AI Disclosure Note:** This notebook was developed with assistance from Claude, an AI model made by Anthropic. The instructor reviewed, edited, and tested all code and text.